In [1]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras

* Login Weights and Biases for Experiment Tracking

In [2]:
wandb.login(anonymous="allow")

wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eromoseleeigbedion to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
sweep_config = {

    'method' : 'grid',
    'metric' : {
        'name': 'val_accuracy',
        'goal': 'maximize'
                },
    'parameters' : {
        'batch_size': {'values': [8,16]},
        'learning_rate': {'values': [0.001,0.0001]},
        'hidden_nodes': {'values': [128,64]},
        'img_size': {'values': [16, 224]},
        'epochs': {'values': [5,10]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="5-flowers-image-classification")

Create sweep with ID: s87nua3j
Sweep URL: https://wandb.ai/eromoseleeigbedion/5-flowers-image-classification/sweeps/s87nua3j


In [4]:
def train():
  with wandb.init() as run:
    config = wandb.config

    # Constants
    IMG_HEIGHT = config.img_size
    IMG_WIDTH = config.img_size
    IMG_CHANNELS = 3
    CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

    def read_and_decode(filename, resize_dims):
        img_bytes = tf.io.read_file(filename)
        img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, resize_dims)
        return img

    def parse_csvline(csv_line):
        record_default = ["", ""]
        filename, label_string = tf.io.decode_csv(csv_line, record_default)

        img = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])

        # Convert label string to an integer index
        label = tf.where(tf.equal(CLASS_NAMES, label_string))[0, 0]

        return img, label

    # Define Train dataset
    train_dataset = (
        tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/train_set.csv")
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Define Validation dataset
    eval_dataset = (
        tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/eval_set.csv")
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Model Architecture
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
        keras.layers.Dense(config.hidden_nodes, activation="relu"),
        keras.layers.Dense(len(CLASS_NAMES), activation="softmax")
    ])

    # Model Optimization
    model.compile(
          optimizer="adam",
          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
          metrics=["accuracy"]
      )

    # Model Training
    model.fit(
      train_dataset,
      validation_data=eval_dataset,
      epochs=config.epochs,
      callbacks=[WandbMetricsLogger(log_freq=5)]
      )

In [5]:
wandb.agent(sweep_id, function=train)

wandb: Agent Starting Run: hz80fs1b with config:
wandb: 	batch_size: 8
wandb: 	epochs: 5
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5


Traceback (most recent call last):
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/tmp/ipython-input-422/1940641507.py", line 60, in train

  File "/usr/local/lib/python3.12/dist-packages/keras/src/ut

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/

...

Traceback (most recent call last):
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/tmp/ipython-input-422/1940641507.py", line 60, in train

  File "/usr/local/lib/python3.12/dist-packages/keras/src/ut

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/

Epoch 1/5


Traceback (most recent call last):
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/tmp/ipython-input-422/1940641507.py", line 60, in train

  File "/usr/local/lib/python3.12/dist-packages/keras/src/ut

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipython-input-422/1940641507.py", line 60, in train
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job

  File "/

* Got error because no Access to the Google Cloud Dataset used for the project

In [13]:
from google.colab import auth
auth.authenticate_user()

In [19]:
import os

# Install gcsfuse
!apt-get install -y fuse
!echo "deb http://packages.cloud.google.com/apt gcsfuse-buster main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!sudo apt-get update
!sudo apt-get install gcsfuse

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fuse is already the newest version (2.9.9-5ubuntu3).
0 upgraded, 0 newly installed, 0 to remove and 103 not upgraded.
deb http://packages.cloud.google.com/apt gcsfuse-buster main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100  1022  100  1022    0     0  13272      0 --:--:-- --:--:-- --:--:-- 13447
OK
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://packages.cloud.google.com/apt gcsfuse-buster InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security In

## Mount GCS bucket

### Subtask:
Mount the `cloud-ml-data` GCS bucket in cell `04f0353b` after the external permissions have been fixed.


**Reasoning**:
The previous steps successfully installed `gcsfuse` and re-authenticated the Google Cloud account. Now, as per the subtask, the next step is to attempt mounting the `cloud-ml-data` GCS bucket to the local directory `gcs_mount` again using the `gcsfuse` command in cell `04f0353b`. This re-execution will confirm if the permissions are now correctly set.



In [31]:
import os
BUCKET_NAME = "cloud-ml-data"
MOUNT_POINT = "gcs_mount"

!mkdir -p $MOUNT_POINT
!gcsfuse --implicit-dirs $BUCKET_NAME $MOUNT_POINT

# Update the dataset paths to use the mounted directory
# This will require modifying the 'train' function in cell zho1fyL2yCwN later.
# For now, this cell just sets up the mount.

{"timestamp":{"seconds":1772058364,"nanos":776032387},"severity":"INFO","message":"Start gcsfuse/3.7.1 (Go version go1.24.11) for app \"\" using mount point: /content/gcs_mount\n","mount-id":"cloud-ml-data-65c00bda"}
{"timestamp":{"seconds":1772058364,"nanos":776070057},"severity":"INFO","message":"GCSFuse Config","mount-id":"cloud-ml-data-65c00bda","CLI Flags":{"implicit-dirs":"true"}}
{"timestamp":{"seconds":1772058364,"nanos":776100677},"severity":"INFO","message":"GCSFuse Config","mount-id":"cloud-ml-data-65c00bda","Full Config":{"AppName":"","CacheDir":"","CloudProfiler":{"AllocatedHeap":true,"Cpu":true,"Enabled":false,"Goroutines":false,"Heap":true,"Label":"gcsfuse-0.0.0","Mutex":false},"Debug":{"ExitOnInvariantViolation":false,"Fuse":false,"Gcs":false,"LogMutex":false},"DisableAutoconfig":false,"DisableListAccessCheck":true,"DummyIo":{"Enable":false,"PerMbLatency":0,"ReaderLatency":0},"EnableAtomicRenameObject":true,"EnableGoogleLibAuth":true,"EnableHns":true,"EnableNewReader":t

## Summary:

### Data Analysis Key Findings

*   **`gcsfuse` Installation Success**: The `gcsfuse` package was successfully installed in cell `d739149f` using `apt-get` from the Google Cloud SDK repository. Version 3.7.1 was installed or confirmed as the newest version.
*   **Persistent Google Cloud Storage Permission Denied Error**: Despite multiple re-authentication attempts, mounting the `cloud-ml-data` GCS bucket in cell `04f0353b` consistently failed with a `PermissionDeniedError`.
*   **Specific Permission Issue Identified**: The error message indicated that the authenticated user, `eromoseleeigbedion@gmail.com`, explicitly "does not have `storage.objects.list` access to the Google Cloud Storage bucket."
*   **External Resolution Required**: The permission issue was identified as an external problem requiring manual intervention in the Google Cloud Console, as the agent provided instructions to grant the `storage.objects.list` permission (e.g., via the "Storage Object Viewer" role) to the user for the `cloud-ml-data` bucket.
*   **Mounting Unsuccessful**: The GCS bucket could not be mounted due to the unresolved `PermissionDeniedError`, preventing further progress on the task.

### Insights or Next Steps

*   The user `eromoseleeigbedion@gmail.com` must be granted the `storage.objects.list` permission for the `cloud-ml-data` bucket in the Google Cloud Console before proceeding.
*   Once permissions are verified, retry mounting the `cloud-ml-data` GCS bucket in cell `04f0353b`, then update dataset paths and re-run the Weights & Biases sweep.
